# Notebook 08 — Non-Parametric Tests

**Module:** 03 — Statistics and Probability  
**Notebook:** 08 of 18  
**Prerequisites:** NB06  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

Parametric tests (t-test, ANOVA) assume the data follow a specific distribution
(usually Normal). When this assumption fails — small samples from skewed distributions,
ordinal data, data with many outliers — non-parametric tests are more reliable.
They work by ranking the data, discarding distributional assumptions.

---
## Step 2 — Intuition

Instead of using the actual values, rank-based tests use the *order* of observations.
Ranks are always uniformly distributed regardless of the original distribution —
so the test's null distribution is distribution-free.

Trade-off: rank tests are slightly less powerful than t-tests *when* the Normal
assumption holds, but much more reliable when it doesn't.

---
## Step 3 — Biological Background

**When non-parametric tests are appropriate in biology:**
- Small sample sizes (n < 10 per group) from unknown distributions
- Ordinal data (e.g. pathology scores: 0, 1, 2, 3)
- Survival data (right-censored: log-rank test)
- NGS coverage distributions: zero-inflated and highly right-skewed
- Comparing two ChIP-seq peak score distributions

**Common non-parametric tests:**

| Test | Parametric analogue | When to use |
|------|--------------------|-----------|
| Mann-Whitney U | Two-sample t-test | Two independent groups, non-normal |
| Wilcoxon signed-rank | Paired t-test | Paired samples, non-normal |
| Kruskal-Wallis | One-way ANOVA | ≥ 3 groups, non-normal |
| Spearman correlation | Pearson | Non-linear monotone relationship |

---
## Step 4 — Mathematical Explanation

**Mann-Whitney U statistic:**

For groups $X$ (size $m$) and $Y$ (size $n$):
$$U = \sum_{i=1}^m \sum_{j=1}^n \mathbf{1}[x_i > y_j]$$

$U$ counts the number of times an observation from $X$ exceeds one from $Y$.
Under $H_0$ (same distribution): $E[U] = mn/2$.

**Equivalence to rank-sum test:**
Rank all $m + n$ observations. Let $R_1$ be the sum of ranks for group 1.
$$U_1 = R_1 - \frac{m(m+1)}{2}$$

**Wilcoxon signed-rank:** rank the absolute differences $|d_i|$, then test
whether positive-rank sum ≈ negative-rank sum.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Mann-Whitney U from scratch
def mann_whitney_u(x: np.ndarray, y: np.ndarray) -> dict:
    """
    Mann-Whitney U test via rank-sum method.

    Returns U, p_value (two-tailed), and ranks.
    """
    x, y = np.asarray(x), np.asarray(y)
    m, n = len(x), len(y)
    combined = np.concatenate([x, y])
    ranks = stats.rankdata(combined)  # handles ties
    R1 = ranks[:m].sum()
    U1 = R1 - m*(m+1)/2
    U2 = m * n - U1
    U = min(U1, U2)
    # Normal approximation for large samples
    mu_U = m * n / 2
    sigma_U = np.sqrt(m * n * (m + n + 1) / 12)
    z = (U - mu_U) / sigma_U
    p_value = 2 * stats.norm.sf(abs(z))
    return dict(U=U, p_value=p_value, z=z)

# Compare two coverage distributions (simulated)
rng = np.random.default_rng(42)
# Coverage at active vs. inactive genomic regions
active   = rng.exponential(scale=30, size=50)  # higher coverage, right-skewed
inactive = rng.exponential(scale=10, size=50)

res_mw = mann_whitney_u(active, inactive)
sp_U, sp_p = stats.mannwhitneyu(active, inactive, alternative='two-sided')

print(f"Mann-Whitney U: custom U={res_mw['U']:.0f}, p={res_mw['p_value']:.4f}")
print(f"                scipy  U={sp_U:.0f}, p={sp_p:.4f}")

# Compare to t-test on skewed data
_, p_ttest = stats.ttest_ind(active, inactive)
print(f"\nt-test p = {p_ttest:.4f}  (less appropriate for exponential data)")

In [ ]:
# Cell 6.2 — When does non-parametric win? Power comparison on skewed data
rng2 = np.random.default_rng(0)
alpha = 0.05
n_sim = 5000

# True effect: one distribution has higher mean
def compare_power(scale_a, scale_b, n, n_sim, rng):
    t_detect, mw_detect = 0, 0
    for _ in range(n_sim):
        a = rng.exponential(scale_a, n)
        b = rng.exponential(scale_b, n)
        if stats.ttest_ind(a, b)[1] < alpha:
            t_detect += 1
        if stats.mannwhitneyu(a, b, alternative='two-sided')[1] < alpha:
            mw_detect += 1
    return t_detect/n_sim, mw_detect/n_sim

for n in [10, 20, 50]:
    t_pwr, mw_pwr = compare_power(30, 15, n, n_sim, rng2)
    print(f"n={n}: t-test power={t_pwr:.2f}, Mann-Whitney power={mw_pwr:.2f}")

In [ ]:
# Cell 6.3 — Kruskal-Wallis (non-parametric ANOVA)
group_a = rng.exponential(10, 20)
group_b = rng.exponential(20, 20)
group_c = rng.exponential(10, 20)  # same as A

H, p_kw = stats.kruskal(group_a, group_b, group_c)
print(f"Kruskal-Wallis: H={H:.4f}, p={p_kw:.4f}")
print(f"(Comparison: one-way ANOVA p={stats.f_oneway(group_a, group_b, group_c)[1]:.4f})")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Coverage distributions + rank transformation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(active, bins=25, alpha=0.6, color="steelblue", density=True, label="Active")
ax.hist(inactive, bins=25, alpha=0.6, color="tomato", density=True, label="Inactive")
ax.set_xlabel("Coverage"); ax.set_ylabel("Density")
ax.set_title(f"Coverage distributions (exponential)\nMann-Whitney p={sp_p:.4f}")
ax.legend()

ax = axes[1]
# Show rank transformation
combined = np.concatenate([active, inactive])
group_labels = ["Active"]*50 + ["Inactive"]*50
ranks_all = stats.rankdata(combined)
ranks_active = ranks_all[:50]
ranks_inactive = ranks_all[50:]
ax.boxplot([ranks_active, ranks_inactive], labels=["Active", "Inactive"],
           patch_artist=True,
           boxprops=dict(facecolor="steelblue", alpha=0.5))
ax.set_ylabel("Rank"); ax.set_title("After rank transformation\n(used in Mann-Whitney)")

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `mann_whitney_u()` from scratch. Verify it against `scipy.stats.mannwhitneyu`.
2. When is the Wilcoxon signed-rank test preferred over the Mann-Whitney U test?
   Apply the Wilcoxon signed-rank test to paired NGS replicate data (simulate 20 pairs).
3. Run the power comparison from Cell 6.2 with **Normal** data (not Exponential).
   Which test has higher power on Normal data? Why?
4. What is the exact null hypothesis of the Mann-Whitney U test? How is it different from
   the t-test null hypothesis?

---
## Quiz — Active Recall

1. What does 'non-parametric' mean? What assumption does it avoid?
2. Describe the Mann-Whitney U test in one sentence — what does it count?
3. When would you choose Mann-Whitney U over the t-test?
4. What is the non-parametric equivalent of ANOVA?
5. What is the Wilcoxon signed-rank test, and when is it used?

---
## Reflection

**Date completed:** ____________________

> *[Do you know when to choose a non-parametric test? Could you explain the rank transformation in an interview?]*

---
*Next: `09_correlation.ipynb`*